<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff; 
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); 
        font-weight: bold; 
        margin-bottom: 10px; 
        font-size: 36px; 
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        🛒 Brazilian E-Commerce Analysis 🔍
    </h1>
</div>

# 📂 Overview

The **Brazilian E-Commerce Public Dataset by Olist** is a real-world dataset from the Olist marketplace in Brazil. It records over **100,000 orders** placed between **2016 and 2018**, including detailed information about **customers, sellers, products, payments, deliveries, and reviews**.

The dataset consists of multiple interconnected CSV files — such as `orders`, `customers`, `sellers`, `order_items`, `payments`, `products`, and `reviews` — enabling end-to-end analysis of the order journey: from purchase and payment to delivery and customer feedback.

Typical use cases include analyzing customer behavior, seller performance, delivery times, payment trends, and building predictive models such as review score prediction or late delivery forecasting.


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #FFFFFF; 
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); 
        font-weight: bold; 
        margin-bottom: 5px; 
        font-size: 28px; 
        font-family: 'Roboto", sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [201]:
# Core data manipulation libraries
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 500) # To display all the columns of dataframe
pd.set_option("max_colwidth", None) # To set the width of the column to maximum

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Load Data</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #FFFFFF; 
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); 
        font-weight: bold; 
        margin-bottom: 5px; 
        font-size: 28px; 
        font-family: "Roboto", sans-serif;
        letter-spacing: 1px;
    ">
        Load Data
    </h1>
</div>


In [202]:
# Load the datasets
df_customers = pd.read_csv("olist_customers_dataset.csv")
df_geolocation = pd.read_csv("olist_geolocation_dataset.csv")
df_order_items = pd.read_csv("olist_order_items_dataset.csv")
df_order_payments = pd.read_csv("olist_order_payments_dataset.csv")
df_order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
df_orders = pd.read_csv("olist_orders_dataset.csv")
df_products = pd.read_csv("olist_products_dataset.csv")
df_sellers = pd.read_csv("olist_sellers_dataset.csv")
df_product_category_name_translation = pd.read_csv("product_category_name_translation.csv")

# Verify shapes
print("Customer Data Shape:", df_customers.shape)
print("\nGeolocation Data Shape:", df_geolocation.shape)
print("\nOrder Items Data Shape:", df_order_items.shape)
print("\nOrder Payment Data Shape:", df_order_payments.shape)
print("\nOrder Review Data Shape:", df_order_reviews.shape)
print("\nOrders Data Shape:", df_orders.shape)
print("\nProducts Data Shape:", df_products.shape)
print("\nSellers Data Shape:", df_sellers.shape)
print("\nProduct Category Name Data Shape:", df_product_category_name_translation.shape)

Customer Data Shape: (99441, 5)

Geolocation Data Shape: (1000163, 5)

Order Items Data Shape: (112650, 7)

Order Payment Data Shape: (103886, 5)

Order Review Data Shape: (99224, 7)

Orders Data Shape: (99441, 8)

Products Data Shape: (32951, 9)

Sellers Data Shape: (3095, 4)

Product Category Name Data Shape: (71, 2)


In [203]:
timestamp_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", 
                  "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in timestamp_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [204]:
# Create column total_value
df_order_items["total_value"] = df_order_items["price"] + df_order_items["freight_value"]

df_order_items_ = df_order_items.groupby(by=["order_id", "product_id", "shipping_limit_date"]).agg(
    product_counts = ("product_id", "count"),
    total_price = ("price", "sum"),
    total_value = ("total_value", "sum")
).reset_index()
df_order_items_.head()

,order_id,product_id,shipping_limit_date,product_counts,total_price,total_value
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,2017-09-19 09:45:35,1,58.90,72.19
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,2017-05-03 11:05:13,1,239.90,259.83
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,2018-01-18 14:48:30,1,199.00,216.87
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,2018-08-15 10:10:18,1,12.99,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,2017-02-13 13:57:51,1,199.90,218.04


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Summary and Report</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #FFFFFF; 
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); 
        font-weight: bold; 
        margin-bottom: 5px; 
        font-size: 28px; 
        font-family: "Roboto", sans-serif;
        letter-spacing: 1px;
    ">
        Summary and Report
    </h1>
</div>


## Customer Analysis
### Customers' Spending By Geolocation

In [205]:
# 1. **Sort (`sort_values`)**: It sorts the `df_geolocation` DataFrame by `geolocation_zip_code_prefix` in ascending order.
# 2. **Group (`groupby`)**: It groups the data by `geolocation_city` and `geolocation_state`.
# 3. **Select the first row of each group (`head(1)`)**: It keeps only the first row of each group after sorting.
data_geo = df_geolocation.sort_values(by="geolocation_zip_code_prefix", ascending=True).groupby(by=["geolocation_city", "geolocation_state"]).head(1)
data_geo.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
771,1001,-23.550498,-46.634338,sao paulo,SP
575,1001,-23.549779,-46.633957,são paulo,SP
10166,1307,-23.556812,-46.657135,sao bernardo do campo,SP
22261,2116,-23.515978,-46.582170,são paulo,RN
50999,3203,-23.216648,-46.861371,jundiaí,SP


In [206]:
# Merge orders with order items
# Filter only delivered orders
# Merge with customer information
# Group by customer city and state
# Aggregate number of orders and total value
# Sort by number of orders in descending order
# Merge with representative geolocation data

data_customers = df_orders.merge(df_order_items, how="inner", on="order_id")\
        .query('order_status == "delivered"')\
        .merge(df_customers, how="inner", on="customer_id")\
        .groupby(by=["customer_city", "customer_state"])\
        .agg(
            orders_count = ("order_id", "nunique"),
            total_value = ("total_value", "sum"))\
        .sort_values(by="orders_count", ascending=False)\
        .merge(data_geo, how="inner", left_on=["customer_city", "customer_state"], right_on=["geolocation_city", "geolocation_state"])
data_customers["location"] = data_customers["geolocation_city"].apply(lambda x: str(x).title()) + ", " + data_customers["geolocation_state"]
data_customers.head(10)

,orders_count,total_value,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,location
0,15045,2107960.17,1001,-23.550498,-46.634338,sao paulo,SP,"Sao Paulo, SP"
1,6601,1111732.21,20010,-22.904775,-43.172688,rio de janeiro,RJ,"Rio De Janeiro, RJ"
2,2697,405950.51,30001,-19.923989,-43.935706,belo horizonte,MG,"Belo Horizonte, MG"
3,2071,345199.05,70002,-15.790439,-47.880655,brasilia,DF,"Brasilia, DF"
4,1489,238459.72,80002,-25.439563,-49.261963,curitiba,PR,"Curitiba, PR"
5,1406,209002.90,13010,-22.905552,-47.049448,campinas,SP,"Campinas, SP"
6,1342,214805.84,90001,-30.028364,-51.230589,porto alegre,RS,"Porto Alegre, RS"
7,1188,207713.30,40010,-12.969912,-38.511830,salvador,BA,"Salvador, BA"
8,1144,157735.65,7010,-23.467578,-46.529161,guarulhos,SP,"Guarulhos, SP"
9,911,116784.58,1307,-23.556812,-46.657135,sao bernardo do campo,SP,"Sao Bernardo Do Campo, SP"


#### Top 10 Selling Geolocation

In [207]:
fig = go.Figure()

fig.add_trace(
    go.Bar (
        x = data_customers["location"].head(10),
        y = data_customers["orders_count"].head(10),
        name = "Number of orders",
        marker_color = "rgb(64,224,208)",
        # hovertemplate = "%{x}" +  "<br>Number of orders: %{y}",
    )
)

fig.add_trace(
    go.Scatter (
        x = data_customers["location"].head(10),
        y = data_customers["total_value"].head(10),
        name = "GMV",
        yaxis = "y2",
        marker_color = "rgb(255,160,122)",
        mode="lines+markers",
        # hovertemplate = "%{x}" + "<br>GMV: $%{y}",
    )
)

fig.update_layout(
    title = dict(text="<b>Top 10 Selling Geolocation<b>",
                 font=dict(size=12, family="Arial", color="black")),
    plot_bgcolor = "white",
    yaxis = dict(side = "left", range = [0, max(data_customers["orders_count"]) + 1000], showgrid = False, zeroline = True, showline = False, showticklabels = False),
    yaxis2 = dict(side = "right", overlaying = "y", showgrid = False, zeroline = False, showline = False, showticklabels = False),
    xaxis = dict(showline = True, linecolor = "rgb(204, 204, 204)", linewidth = 2),
    legend = dict(orientation="h", x=0.8, y = 1.1),
    annotations=[dict(text="Created By Thuan Dao.", xref="paper", yref="paper", x=1, y=-.45, 
    showarrow=False, font=dict(size=10, color="gray", family="Arial"))],
    hovermode = "x unified"
)


<font color = '#e3f56e'>
Insight:

- **São Paulo, SP** leads in both order volume and GMV, marking it as a key market.
- **Rio de Janeiro** and **Belo Horizonte** show strong sales performance and revenue contribution.
- **Brasília, DF** has fewer orders but high GMV, indicating higher average order value.
- Variations between order count and GMV across cities reflect different consumer behaviors.

</font>

#### Consumers spending by region

In [208]:
# Define a function to perform Min-Max scaling for a single value
# x: the value to scale
# data: the DataFrame containing the column
# col: the column name to scale
def scaling(x, data, col):
    _min = min(data[col])  # find the minimum value in the column
    _max = max(data[col])  # find the maximum value in the column
    return (x - _min)/(_max - _min)  # apply Min-Max scaling formula

# Apply scaling to the "orders_count" column
# Create a new column "orders_count_scaling" with scaled values
data_customers["orders_count_scaling"] = data_customers["orders_count"].apply(
    lambda x: scaling(x, data_customers, "orders_count")
)

# Apply scaling to the "total_value" column
# Create a new column "total_value_scaling" with scaled values
data_customers["total_value_scaling"] = data_customers["total_value"].apply(
    lambda x: scaling(x, data_customers, "total_value")
)

# Calculate the geographic center of all customers
# lat_center: average latitude
lat_center = np.mean(data_customers["geolocation_lat"])
# lon_center: average longitude
lon_center = np.mean(data_customers["geolocation_lng"])

In [209]:
data_customers.loc [0:1, "total_value_scaling"] = data_customers.loc [0:1, "total_value_scaling"]/2
token = open("mapbox_token").read().strip()

fig = go.Figure(go.Scattermapbox(
        lat = data_customers["geolocation_lat"],
        lon = data_customers["geolocation_lng"],
        mode = "markers",
        marker = go.scattermapbox.Marker (
            size = data_customers["total_value_scaling"]*300,
            sizemin = min(data_customers["total_value"]/12),
            color = data_customers["orders_count_scaling"]*100,
            cmin = min(data_customers["orders_count"])*100,
            colorscale = "teal"
        ),
        textposition="top right",
        hoverinfo= "text",
        hovertext = (
            data_customers["location"].astype(str) +  "<br>" +
            "Number of orders: " + data_customers["orders_count"].astype(str) + "<br>" +
            "GMV: $" + round(data_customers["total_value"], 2).astype(str)
        ),
    )
)

fig.update_layout(
    autosize = True,
    margin = {"r":5,"t":5,"l":5,"b":5},
    hovermode = "closest",
    showlegend = False,
    title = dict(
        text = "<b>Consumers spending by region<b>",
        font = dict(size=12, family="Arial", color="white"),
        x = 0.01,
        y = 0.95
    ),
    mapbox = dict(accesstoken = token, bearing = 0, center = go.layout.mapbox.Center(lat = lat_center, lon = lon_center), 
                  pitch = 0, zoom = 3, style = "dark")
)
fig.show()

#### Consumers' spending by state

In [210]:
import json
from urllib.request import urlopen

brazil_geojson_url = "https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/brazil-states.geojson"

with urlopen(brazil_geojson_url) as response:
    brazil_states = json.load(response)

data_customers_states = data_customers.groupby (by = "geolocation_state")[["orders_count", "total_value"]].sum().reset_index()

state_id_map = {}
for feature in brazil_states["features"]:
    feature["id"] = feature["properties"]["sigla"]
    state_id_map[feature["id"]] =  feature["properties"]["name"]

state_id_map

{'AC': 'Acre',
 'AL': 'Alagoas',
 'AM': 'Amazonas',
 'AP': 'Amapá',
 'BA': 'Bahia',
 'CE': 'Ceará',
 'ES': 'Espírito Santo',
 'GO': 'Goiás',
 'MA': 'Maranhão',
 'MG': 'Minas Gerais',
 'MS': 'Mato Grosso do Sul',
 'MT': 'Mato Grosso',
 'PA': 'Pará',
 'PB': 'Paraíba',
 'PE': 'Pernambuco',
 'PI': 'Piauí',
 'PR': 'Paraná',
 'RJ': 'Rio de Janeiro',
 'RN': 'Rio Grande do Norte',
 'RO': 'Rondônia',
 'RR': 'Roraima',
 'RS': 'Rio Grande do Sul',
 'SC': 'Santa Catarina',
 'SE': 'Sergipe',
 'SP': 'São Paulo',
 'TO': 'Tocantins',
 'DF': 'Distrito Federal'}

In [211]:
# Map state codes/names in "geolocation_state" column to corresponding IDs using state_id_map
# Then convert the mapped IDs to string type
data_customers_states["name"] = data_customers_states["geolocation_state"].apply(
    lambda x: state_id_map[x]  # map each state to its corresponding ID
).astype(str)  # convert the result to string

display(data_customers_states.head())

,geolocation_state,orders_count,total_value,name
0,AC,80,19575.33,Acre
1,AL,397,94172.49,Alagoas
2,AM,145,27585.47,Amazonas
3,AP,67,16141.81,Amapá
4,BA,3242,587373.55,Bahia


In [212]:
def world_map_transaction(df=data_customers_states, feature = "name"):
    import folium
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

    df_location = df.copy()

    # Calculate the approximate latitude/ longitude coordinates for unique state in the data
    unique_state = df_location[feature].unique()
    state_coords = {}

    geolocator = Nominatim(user_agent="location_mapper")
    geocode = RateLimiter(
        geolocator.geocode,
        min_delay_seconds=1,
        max_retries=3,
        error_wait_seconds=3,
        swallow_exceptions=False
    )

    print(f"🔍 Fetching coordinates for {len(unique_state)} unique state...")

    # Get approximate coordinates for each unique state
    for state in unique_state:
        try:
            loc = geocode(f"{state}, Brazil", timeout=15)
            if loc:
                state_coords[state] = (loc.latitude, loc.longitude)
            else:
                print(f"⚠️ Coordinates not found for {state}")
        except Exception as e:
            print(f"❌ Error fetching coordinates for {state}: {e}")

    # Add the coordinates to the DataFrame
    df_location["name"] = df_location["name"].astype(str)
    df_location["Coordinates"] = df_location["name"].map(state_coords)
    df_location = df_location[
        df_location["Coordinates"].apply(lambda x: isinstance(x, (list, tuple)) and len(x) == 2)
    ]

    # Iniitialize the Folium map centered at an approximate central point
    if len(state_coords) > 0:
        initial_coords = list(state_coords.values())[0]
    else:
        print("⚠️ No coordinates found — initializing map at (0,0)")
        initial_coords = [0, 0]

    mymap = folium.Map(location=initial_coords, zoom_start=5, tiles="CartoDB dark_matter")

    for _, row in df_location.iterrows():
        coords = row["Coordinates"]

        tooltip_html = f"""
        <div style='font-size:13px; line-height:1.5'>
            <b>State:</b> {row['name']}<br>
            <b>Sigla:</b> {row['geolocation_state']}<br>
            <b>Number Order:</b> {row['orders_count']}<br>
            <b>GMV:</b> ${row['total_value']:,.2f}
        </div>
        """

        folium.Marker(
            location=coords,
            tooltip=folium.Tooltip(tooltip_html, sticky=True),
            icon=folium.Icon(color="lightgray", icon="info-sign")
        ).add_to(mymap)

    map_path = f"./saved-map-brazil-olist.html"
    mymap.save(map_path)
    print(f"✅ Map saved successfully: {map_path}")

    return mymap

world_map_transaction()

🔍 Fetching coordinates for 27 unique state...
✅ Map saved successfully: ./saved-map-brazil-olist.html


### Customers Returned and New Customers

In [213]:
# Merge orders with order items to get product-level details
# Keep only orders with status "delivered"
# Then merge with customers to get customer information
# Create a new column "year_month" from order purchase timestamp
# Select only relevant columns for time series analysis
data_timeseries = (
    df_orders
    .merge(df_order_items_, how="left", on="order_id")  # join orders with order items
    .query('order_status == "delivered"')  # filter only delivered orders
    .merge(df_customers, how="left", on="customer_id")  # join with customers
    .assign(year_month = lambda x: x["order_purchase_timestamp"].dt.to_period("M"))  # extract year-month period
    [["order_id", "customer_unique_id", "order_purchase_timestamp", "year_month", "product_counts", "total_price", "total_value"]]  # select relevant columns
)

# Create an order index per customer based on purchase timestamp
# Rank orders chronologically for each customer
data_timeseries["order_index"] = data_timeseries.groupby("customer_unique_id")["order_purchase_timestamp"]\
                                               .rank(method="first", ascending=True)

display(data_timeseries.head())

,order_id,customer_unique_id,order_purchase_timestamp,year_month,product_counts,total_price,total_value,order_index
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,2017-10,1.0,29.99,38.71,2.0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,2018-07,1.0,118.70,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,2018-08,1.0,159.90,179.12,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,2017-11,1.0,45.00,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,2018-02,1.0,19.90,28.62,1.0


In [214]:
# Create a monthly summary of new vs returning customers
data_customer_timeseries = (
    # Step 1: Aggregate new customers (first order of each customer)
    data_timeseries.query("order_index == 1")  # select only first orders
    .groupby("year_month")
    .agg(
        new_customers_count=("customer_unique_id", "nunique"),  # number of new customers
        new_customers_value=("total_value", "sum"),  # total value of new customers
    )
    # Step 2: Merge with returning customers (orders after first)
    .merge(
        right=(
            data_timeseries.query("order_index > 1")  # select repeat orders
            .groupby("year_month")
            .agg(
                return_customers_count=("customer_unique_id", "nunique"),  # number of returning customers
                return_customers_value=("total_value", "sum"),  # total value from returning customers
            )
        ),
        how="left",  # keep all months from new customers
        on="year_month"
    )
    .fillna(0)  # fill months with no returning customers with 0
    .reset_index()  # reset index for clean DataFrame
)

display(data_customer_timeseries.head())

,year_month,new_customers_count,new_customers_value,return_customers_count,return_customers_value
0,2016-09,1,143.46,0.0,0.00
1,2016-10,262,44687.95,12.0,1802.71
2,2016-12,1,19.62,0.0,0.00
3,2017-01,717,121229.75,46.0,6252.62
4,2017-02,1628,262457.36,63.0,8781.96


In [215]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x = data_customer_timeseries["year_month"].astype(str),
        y = data_customer_timeseries["new_customers_count"],
        name = "Number of new customers",
        marker_color = "mediumaquamarine"
    )
)

fig.add_trace(
    go.Bar(
        x = data_customer_timeseries["year_month"].astype(str),
        y = data_customer_timeseries["return_customers_count"],
        name = "Number of returned customers",
        marker_color = "powderblue"
    )
)

fig.add_trace(
    go.Scatter(
        x = data_customer_timeseries["year_month"].astype(str),
        y = data_customer_timeseries["new_customers_value"],
        name = "Revenue from new customers",
        yaxis = "y2",
        marker_color = "indianred",
        mode="lines+markers"
    )
)

fig.add_trace(
    go.Scatter(
        x = data_customer_timeseries["year_month"].astype(str),
        y = data_customer_timeseries["return_customers_value"],
        name = "Revenue from returned customers",
        yaxis = "y2",
        marker_color = "sandybrown",
        mode="lines+markers"
    )
)

fig.update_layout(
    title = dict(text="<b>New Customers And Return Customers By Month<b>",
                 font=dict(size=12, family="Arial", color="black")),
    plot_bgcolor = "white",
    barmode = "stack",
    yaxis = dict(side = "left", showgrid = False, zeroline = True, showline = False, showticklabels = False),
    yaxis2 = dict(side = "right", overlaying = "y", showgrid = False, zeroline = False, showline = False, showticklabels = False),
    xaxis = dict(showline = True, linecolor = "rgb(204, 204, 204)", linewidth = 2),
    legend = dict(orientation="h"),
    hovermode = "x unified",
    annotations=[dict(text="Created By Thuan Dao.", xref="paper", yref="paper", x=1.05, y=-0.25, 
    showarrow=False, font=dict(size=10, color="gray", family="Arial"))]
)


<font color = '#e3f56e'>
Insight:

- In general, the majority of customers to this e-commerce sites are new customers with once-in-a-lifetime purchase. From 2017 onwards, it has done a great job in attracting new customers and the expenditure from this group attributed a large portion.<br>
- Returned customers number, on the other hand, stabalised over the periods, and spending from this group is also negligible.
</font>

### Customers Lifetime Value

In [216]:
# Aggregate customer-level metrics from the timeseries data
data_orders = (
    data_timeseries
    .groupby("customer_unique_id")  # group data by each unique customer
    .agg(
        last_purchase_date=("order_purchase_timestamp", "max"),  # most recent purchase date
        order_count=("order_id", "nunique"),  # total number of orders
        quantity=("product_counts", "sum"),  # total products purchased
        total_price=("total_price", "sum"),  # total price across all orders
        total_value=("total_value", "sum")   # total value across all orders
    )
    .sort_values(by="order_count", ascending=False)  # sort customers by number of orders, descending
    .reset_index()  # reset index to turn customer_unique_id into a column
)

data_orders.head()

,customer_unique_id,last_purchase_date,order_count,quantity,total_price,total_value
0,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,15,15.0,714.63,879.27
1,3e43e6105506432c953e165fb2acf44c,2018-02-27 18:36:39,9,14.0,1000.85,1172.67
2,6469f99c1f9dfae7733b25662e7f1782,2018-06-28 00:43:34,7,9.0,664.20,758.83
3,1b6c7548a2a1f9037c1fd3ddfed95f33,2018-02-14 13:22:12,7,9.0,809.21,959.01
4,ca77025e7201e3b30c44b472ff346268,2018-06-01 11:38:29,7,12.0,806.61,1122.72


In [217]:
# Calculate Average Order Value (AOV) for each customer
# AOV = total value of all orders / number of orders
data_orders["AOV"] = data_orders["total_value"] / data_orders["order_count"]

# Calculate overall purchase frequency across all customers
# purchase_freq = total number of orders / total number of unique customers
purchase_freq = data_orders["order_count"].sum() / len(data_orders)

# Calculate repeat rate: proportion of customers who made more than one order
repeat_rate = data_orders[data_orders["order_count"] > 1].shape[0] / data_orders.shape[0]

# Calculate churn rate: proportion of customers who did not make a repeat purchase
churn_rate = 1 - repeat_rate

# Estimate profit margin (assuming 10% profit on total price)
data_orders["profit_margin"] = data_orders["total_price"] * 0.1

# Calculate Customer Lifetime Value (CLV)
# CLV = (AOV × purchase frequency) / churn rate × 100
# The multiplier (×100) scales CLV to a more interpretable range
data_orders["CLV"] = (data_orders["AOV"] * purchase_freq) / churn_rate * 100

q33 = data_orders["CLV"].quantile(0.33)
q66 = data_orders["CLV"].quantile(0.66)

# Function to classify customers
def clv_segment(x):
    if x <= q33:
        return "Low Value"
    elif x <= q66:
        return "Medium Value"
    else:
        return "High Value"

# Add Segment column
data_orders["CLV_Segment"] = data_orders["CLV"].apply(clv_segment)

# Display top customers with highest CLV (VIP customers)
print("Top 10 High-Value Customers:")
display(data_orders.sort_values(by="CLV", ascending=False).reset_index(drop=True).head(10))

Top 10 High-Value Customers:


,customer_unique_id,last_purchase_date,order_count,quantity,total_price,total_value,AOV,profit_margin,CLV,CLV_Segment
0,0a0a92112bd4c708ca5fde585afaa872,2017-09-29 15:24:52,1,8.0,13440.00,13664.08,13664.08,1344.000,1.455750e+06,High Value
1,763c8b1c9c68a0229c42c9fc6f662b93,2018-07-15 14:49:44,1,4.0,7160.00,7274.88,7274.88,716.000,7.750542e+05,High Value
2,dc4802a71eae9be1dd28f5d788ceb526,2017-02-12 20:37:36,1,1.0,6735.00,6929.31,6929.31,673.500,7.382378e+05,High Value
3,459bef486812aa25204be022145caa62,2018-07-25 18:10:17,1,1.0,6729.00,6922.21,6922.21,672.900,7.374813e+05,High Value
4,ff4159b92c40ebe40454e3e6a7c35ed6,2017-05-24 18:14:34,1,1.0,6499.00,6726.66,6726.66,649.900,7.166478e+05,High Value
5,4007669dec559734d6f53e029e360987,2017-11-24 11:03:35,1,6.0,5934.60,6081.54,6081.54,593.460,6.479177e+05,High Value
6,eebb5dda148d3893cdaf5b5ca3040ccb,2017-04-18 18:50:13,1,1.0,4690.00,4764.34,4764.34,469.000,5.075853e+05,High Value
7,48e1ac109decbb87765a3eade6854098,2018-06-22 12:23:19,1,1.0,4590.00,4681.78,4681.78,459.000,4.987895e+05,High Value
8,edde2314c6c30e864a128ac95d6b2112,2018-08-03 21:10:16,1,1.0,4399.87,4513.32,4513.32,439.987,4.808420e+05,High Value
9,a229eba70ec1c2abef51f04987deb7a5,2018-05-31 22:57:07,1,2.0,4400.00,4445.50,4445.50,440.000,4.736166e+05,High Value


In [218]:
clv_group_dist = (data_orders["CLV_Segment"].value_counts().reset_index())

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'xy'}]])
fig.add_trace(
    go.Pie(
        labels= clv_group_dist["CLV_Segment"],
        values=clv_group_dist["count"],
        textinfo="label+percent",
        hovertemplate="Group: %{label}<br>Number of Customers: %{value}",
        marker=dict(colors=["#E74C3C", "#F1C40F", "#27AE60"], line=dict(color="white", width=2)),
        name="CLV Segment Distribution",
        hole=0.8
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x = clv_group_dist["CLV_Segment"].astype(str),
        y = clv_group_dist["count"],
        text=clv_group_dist["count"],
        textposition="auto",
        marker=dict(color=["#E74C3C", "#F1C40F", "#27AE60"], line=dict(width=1, color="white")),
        name="Count Plot of CLV Segment Distribution"
    ),
    row=1, col=2
)

fig.update_layout(
    title = dict(text="<b>Customer Lifetime Value (CLV) Segmentation Overview<b>",
                 font=dict(size=12, family="Arial", color="gray")),
    plot_bgcolor = "white",
    xaxis = dict(showline = True, linecolor = "rgb(204, 204, 204)", linewidth = 2),
    yaxis = dict(side = "left", showgrid = False, zeroline = True, showline = False, showticklabels = False),
    showlegend=False,
    annotations=[dict(text="Created By Thuan Dao.", xref="paper", yref="paper", x=1, y=-.45, 
    showarrow=False, font=dict(size=10, color="gray"))]
)

fig.show()

<font color = '#e3f56e'>
Insight:

- **High Value Customers** make up **34%** of the base, with the largest count (**31,736 customers**) — key targets for retention and upselling.
- **Medium Value** and **Low Value** segments are nearly equal in size (~33%), each with over **30,800 customers** — ideal for nurturing and growth strategies.
- The balanced distribution across segments suggests opportunities for personalized engagement and tiered marketing approaches.
</font>

### Customers Segmentation Using RFM Model

In [219]:
# Create a new DataFrame for RFM analysis with selected columns
df_rfm = (
    data_orders[["customer_unique_id", "last_purchase_date", "order_count", "total_price"]].copy()
    .sort_values(by="last_purchase_date", ascending=True)  # sort by last purchase date ascending
)

# Rename columns to standard RFM terminology
df_rfm.columns = ["customer_unique_id", "last_purchase_date", "freq", "monetary"]

# Define the reference date for recency calculation
# +1 day is added to avoid Recency = 0 for customers who transacted on the latest date
recency_date = df_rfm["last_purchase_date"].max() + pd.Timedelta(days=1)

# Calculate recency (number of days since last purchase)
df_rfm["recency"] = df_rfm["last_purchase_date"].apply(
    lambda x: (recency_date - x).days
).fillna(0).astype(int)

display(df_rfm.head())

,customer_unique_id,last_purchase_date,freq,monetary,recency
47882,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38,1,134.97,714
18267,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:50,1,21.90,695
36173,61db744d2f835035a5625b59350c6b63,2016-10-03 21:13:36,1,36.49,695
51400,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03 22:06:03,1,119.90,695
49410,87776adb449c551e74c13fc34f036105,2016-10-03 22:31:31,1,29.99,695


In [220]:
# Recency scoring: lower recency = better (more recent purchase)
def r_score(value, r_quartiles):
    if value >= r_quartiles[0.8]:
        return 1
    elif value >= r_quartiles[0.6]:
        return 2
    elif value >= r_quartiles[0.4]:
        return 3
    elif value >= r_quartiles[0.2]:
        return 4
    else:
        return 5

# Frequency scoring: higher frequency = better
def f_score(value, f_quartiles):
    if value >= f_quartiles[0.8]:
        return 5
    elif value >= f_quartiles[0.6]:
        return 4
    elif value >= f_quartiles[0.4]:
        return 3
    elif value >= f_quartiles[0.2]:
        return 2
    else:
        return 1
    
# Monetary scoring: higher monetary value = better
def m_score(value, m_quartiles):
    if value >= m_quartiles[0.8]:
        return 5
    elif value >= m_quartiles[0.6]:
        return 4
    elif value >= m_quartiles[0.4]:
        return 3
    elif value >= m_quartiles[0.2]:
        return 2
    else:
        return 1

# Compute quartiles for Recency, Frequency, Monetary
r_quartiles = df_rfm["recency"].quantile([0.2, 0.4, 0.6, 0.8])
f_quartiles = df_rfm["freq"].quantile([0.2, 0.4, 0.6, 0.8])
m_quartiles = df_rfm["monetary"].quantile([0.2, 0.4, 0.6, 0.8])

# Apply scoring functions to each customer
df_rfm["R_Score"] = df_rfm["recency"].apply(lambda x: r_score(x, r_quartiles))
df_rfm["F_Score"] = df_rfm["freq"].apply(lambda x: f_score(x, f_quartiles))
df_rfm["M_Score"] = df_rfm["monetary"].apply(lambda x: m_score(x, m_quartiles)) 

# Combine R, F, M scores into a single RFM score as a string
df_rfm["RFM_Score"] = (
    df_rfm["R_Score"].astype(str) + 
    df_rfm["F_Score"].astype(str) + 
    df_rfm["M_Score"].astype(str)
)

display(df_rfm.head())

,customer_unique_id,last_purchase_date,freq,monetary,recency,R_Score,F_Score,M_Score,RFM_Score
47882,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38,1,134.97,714,1,5,4,154
18267,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:50,1,21.90,695,1,5,1,151
36173,61db744d2f835035a5625b59350c6b63,2016-10-03 21:13:36,1,36.49,695,1,5,1,151
51400,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03 22:06:03,1,119.90,695,1,5,4,154
49410,87776adb449c551e74c13fc34f036105,2016-10-03 22:31:31,1,29.99,695,1,5,1,151


In [221]:
# define a function to assign segments based on the RFM scores
def rfm_segment(score):
    if score in ["555", "554", "545", "544", "545", "455", "445"]:
        return "Champions"
    elif score in ["543", "444", "435", "355", "354", "345", "344", "335"]:
        return "Loyal"
    elif score in ["553", "551", "552", "541", "542", "533", "532", "531", "452", "451", "442", "441", 
                   "431", "453", "433", "432", "423", "353", "352", "351", "342", "341", "333", "323"]:
        return "Potential Loyalist"
    elif score in ["525", "524", "523", "522", "521", "515", "514", "513", "425", "424", "413", "414", 
                   "415", "315", "314", "313"]:
        return "Promising"
    elif score in ["512", "511", "422", "421", "412", "411", "311"]:
        return "New Customers"
    elif score in ["535", "534", "443", "434", "343", "334", "325", "324"]:
        return "Need Attention"
    elif score in ["331", "321", "312", "221", "213", "231", "241", "251"]:
        return "About To Sleep"
    elif score in ["255", "254", "245", "244", "243", "252", "243", "242", "235", "234", "225", "224",
                   "153", "152", "145", "143", "142", "135", "134", "133", "125", "124"]:
        return "At Risk"
    elif score in ["155", "154", "144", "214", "215", "115", "114", "113"]:
        return "Cannot Lose Them"
    elif score in ["332", "322", "233", "232", "223", "222", "132", "123", "122", "212", "211"]:
        return "Hibernating Customers"
    elif score in ["111", "112", "121", "131", "141", "151"]:
        return "Lost Customers"
    else:
        return "Other"

df_rfm["Segment"] = df_rfm["RFM_Score"].apply(rfm_segment)
display(df_rfm.head())

,customer_unique_id,last_purchase_date,freq,monetary,recency,R_Score,F_Score,M_Score,RFM_Score,Segment
47882,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38,1,134.97,714,1,5,4,154,Cannot Lose Them
18267,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:50,1,21.90,695,1,5,1,151,Lost Customers
36173,61db744d2f835035a5625b59350c6b63,2016-10-03 21:13:36,1,36.49,695,1,5,1,151,Lost Customers
51400,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03 22:06:03,1,119.90,695,1,5,4,154,Cannot Lose Them
49410,87776adb449c551e74c13fc34f036105,2016-10-03 22:31:31,1,29.99,695,1,5,1,151,Lost Customers


In [222]:
# Aggregate RFM data by customer Segment
data_rfm_summary = (
    df_rfm
    .groupby("Segment")  # group by customer segment
    .agg(
        customer_count=("Segment", "count"),  # number of customers in each segment
        total_monetary=("monetary", "sum")    # total monetary value in each segment
    )
    .reset_index()  # reset index for clean DataFrame
)

# Calculate total monetary value across all segments
total_monetary_all = data_rfm_summary["total_monetary"].sum()

# Calculate the percentage contribution of each segment to total monetary value
data_rfm_summary["total_monetary_percent"] = round(
    (data_rfm_summary["total_monetary"] / total_monetary_all) * 100, 2
)

# Scale the total monetary values to range [0,1] using the previously defined scaling function
data_rfm_summary["total_monetary_scaling"] = data_rfm_summary["total_monetary"].apply(
    lambda x: scaling(x, data_rfm_summary, "total_monetary")
)

display(data_rfm_summary.head())

,Segment,customer_count,total_monetary,total_monetary_percent,total_monetary_scaling
0,About To Sleep,3509,90078.94,0.68,0.000000
1,At Risk,19143,2830407.45,21.41,0.778986
2,Cannot Lose Them,7290,2001949.19,15.14,0.543482
3,Champions,11432,3607892.80,27.29,1.000000
4,Lost Customers,3873,97266.38,0.74,0.002043


In [223]:
# Create a dictionary to map each customer segment to a color based on scaled total monetary value
color = {}

# Iterate over each segment and its scaled total monetary value
for ele in data_rfm_summary[["Segment", "total_monetary_scaling"]].to_dict("records"):
    if ele["total_monetary_scaling"] <= 0.1:
        color[ele["Segment"]] = "#ffcab3"  # very low contribution
    elif ele["total_monetary_scaling"] <= 0.3:
        color[ele["Segment"]] = "#ffb999"  # low contribution
    elif ele["total_monetary_scaling"] <= 0.5:
        color[ele["Segment"]] = "#ff9e80"  # medium contribution
    elif ele["total_monetary_scaling"] <= 0.8:
        color[ele["Segment"]] = "#ff9566"  # high contribution
    else:
        color[ele["Segment"]] = "#ff6119"  # very high contribution

In [224]:
def annotation (x, y, name):
    monetary = round(float(data_rfm_summary[data_rfm_summary["Segment"] == name]["total_monetary"]), 2)
    monetary_percent = round(float(data_rfm_summary[data_rfm_summary["Segment"] == name]["total_monetary_percent"]), 2)
    customers_count = int(data_rfm_summary[data_rfm_summary["Segment"] == name]["customer_count"])

    text = f"<b>{name}</b><br>Total Customers: {customers_count}<br>Total Monetary: {monetary} ({monetary_percent}%)"

    return fig.add_annotation (
        x = x, y = y, xref = "x domain", yref= "y domain", font = dict(color = "black", size = 11),
        text = text, align= "left", xanchor = "left", showarrow = False)

def property (x, y, name):
    return go.Scatter (
        x = x,
        y = y,
        fill = "toself",
        fillcolor = color[name],
        hoveron = "fills",
        hoverinfo = "text",
        line_color = "white",
        mode = "lines+text",
        name = name
    )

In [225]:
fig = go.Figure()

fig.add_trace(
    property(
        x = [0, 0, 2, 2],
        y = [4, 5, 5, 4],
        name = "Lost Customers"
    )
)

fig.add_trace(
    property (
        x = [0, 0, 2, 2],
        y = [0, 2, 2, 0],
        name = "At Risk",
    )
)

fig.add_trace(
    property (
        x = [0, 0, 2, 2],
        y = [2, 4, 4, 2],
        name = "Cannot Lose Them"
    )
)

fig.add_trace(
    property (
        x = [2, 2, 5, 5],
        y = [3, 5, 5, 3],
        name = "Champions"
    )
)

fig.add_trace(
    property (
        x = [2, 2, 3.5, 3.5],
        y = [2, 3, 3, 2],
        name="About To Sleep"
    )
)

fig.add_trace(
    property (
        x = [2, 2, 3.5, 3.5],
        y = [0, 2, 2, 0],
        name = "Loyal",
    )
)


fig.add_trace(
    property (
        x = [3.5, 3.5, 5, 5],
        y = [1, 3, 3, 1],
        name = "Potential Loyalist",
    )
)

fig.add_trace(
    property (
        x = [3.5, 3.5, 5, 5],
        y = [0, 1, 1, 0],
        name = "Other",
    )
)

annotation (x = 0.01, y = 1, name = "Lost Customers")
annotation (x = 0.01, y = 0.05, name = "At Risk")
annotation (x = 0.01, y = 0.5, name = "Cannot Lose Them")
annotation (x = 0.41, y = 1.0, name = "Champions")
annotation (x = 0.41, y = 0.50, name = "About To Sleep")
annotation (x = 0.41, y = 0.01, name = "Loyal")
annotation (x = 0.71, y = 0.51, name = "Potential Loyalist")
annotation (x = 0.71, y = 0.01, name = "Other")

fig.update_layout(
    title = dict(text="<b>Customer Segmentation by RFM Score<b>",
                 font=dict(size=12, family="Arial", color="black")),
    plot_bgcolor = "white",
    xaxis = dict (showline = True, range = [0, 5],  rangemode = "nonnegative"),
    yaxis = dict (showline = True, range = [0, 5], rangemode = "nonnegative", tickmode="array", tickvals=[1, 2, 3, 4, 5]),
    showlegend = False
)

fig.add_annotation(text="Created By Thuan Dao.", xref="paper", yref="paper", x=1, y=-.2, showarrow=False, font=dict(size=10, color="gray", family="Arial"))

In [226]:
data_rfm_summary = data_rfm_summary.sort_values(by="total_monetary", ascending=False).reset_index()
# Calculate cumulative percentage
data_rfm_summary["Cumulative_%"] = data_rfm_summary["total_monetary"].cumsum() / data_rfm_summary["total_monetary"].sum() * 100
display(data_rfm_summary.head())

,index,Segment,customer_count,total_monetary,total_monetary_percent,total_monetary_scaling,Cumulative_%
0,3,Champions,11432,3607892.80,27.29,1.000000,27.288079
1,1,At Risk,19143,2830407.45,21.41,0.778986,48.695694
2,2,Cannot Lose Them,7290,2001949.19,15.14,0.543482,63.837315
3,5,Loyal,7524,1891409.09,14.31,0.512060,78.142873
4,7,Potential Loyalist,32849,1817609.29,13.75,0.491081,91.890251


In [227]:
def color_gradient(n_colors=5, tone="rocket_r"):
    """
    Generate a gradient or categorical color palette for Plotly.
    tone: Name of the seaborn palette, e.g. 'rocket', 'viridis', 'Dark2', 'crest', 'mako', etc.
    """
    try:
        # If the palette is continuous (sequential), create a colormap
        cmap = sns.color_palette(tone, as_cmap=True)
        positions = np.linspace(0, 1, n_colors)
        colors = [sns.utils.rgb2hex(cmap(p)) for p in positions]
    except Exception:
        # If the palette is discrete (qualitative), use a standard seaborn palette
        colors = sns.color_palette(tone, n_colors).as_hex()
    return colors

In [228]:
fig = go.Figure()

fig.add_trace(
    go.Bar (
        x = data_rfm_summary["Segment"].astype(str),
        y = data_rfm_summary["total_monetary"].round(2),
        name = "Total Monetary",
        # text="$" + data_rfm_summary["total_monetary"].round(2).astype(str),
        # textposition="auto",
        marker_color=color_gradient(n_colors=len(data_rfm_summary), tone="crest"),
        hovertemplate="<b>%{x}</b><br>Total Monetary: $%{y: ,.2f}<extra></extra>" 
    )
)

fig.add_trace(
    go.Scatter (
        x = data_rfm_summary["Segment"].astype(str),
        y = data_rfm_summary["Cumulative_%"],
        name = "Cumulative %",
        mode="lines+markers+text",
        yaxis = "y2",
        # text=data_rfm_summary["Cumulative_%"].round(2).astype(str) + "%",
        marker_color = "rgb(255,160,122)",
        # textposition="top center",
        line=dict(color="rgb(192, 57, 43)", width=3),
        hovertemplate="<b>%{x}</b><br>Cumulative: %{y:.1f}%<extra></extra>"
    )
)

fig.update_layout(
    title = dict(text="<b>Pareto Chart of Customer Segments by Monetary Value<b>",
                 font=dict(size=12, family="Arial", color="black")),
    plot_bgcolor = "white",
    yaxis = dict(side = "left", showgrid = False, zeroline = True, showline = False, showticklabels = True),
    yaxis2 = dict(side = "right", overlaying = "y", showgrid = False, zeroline = False, showline = False, showticklabels = True, ticksuffix="%", range=[0, 100]),
    xaxis = dict(showline = True, linecolor = "rgb(204, 204, 204)", linewidth = 2),
    annotations=[dict(text="Created By Thuan Dao.", xref="paper", yref="paper", x=1, y=-.45, 
    showarrow=False, font=dict(size=10, color="gray", family="Arial"))],
    showlegend=False
)

fig.add_annotation(text="Created By Thuan Dao.", xref="paper", yref="paper", x=1, y=-.2, showarrow=False, font=dict(size=10, color="gray"))

<font color = '#e3f56e'>
Insight

- **Champions** and **Potential Loyalists** are the most valuable segments, contributing **6.62%** and **21.59%** of total revenue respectively. These groups should be prioritized for retention and engagement.
- **Other** has the largest customer base (77,803 customers), contributing **17.64%** of revenue — a promising segment for deeper analysis and nurturing.
- Segments like **Lost Customers**, **About To Sleep**, and **At Risk** have very low monetary value (under 2%), and may require reactivation strategies.
- **Cannot Lose Them** and **Loyal** are stable and consistent contributors, ideal for ongoing relationship management.
</font>

## Product Analysis

In [229]:
# Aggregate product-level data and summarize by category
data_products = (
    # Step 1: Aggregate total product counts and total price by product_id
    df_order_items_.groupby("product_id")[["product_counts", "total_price"]].sum()
    # Step 2: Merge with product details (e.g., category) from df_products
    .merge(df_products, how="left", on="product_id")
    # Step 3: Aggregate again by product category
    .groupby("product_category_name")[["product_counts", "total_price"]].sum()
    # Step 4: Sort categories by total product counts in descending order
    .sort_values(by="product_counts", ascending=False)
    .reset_index()  # Reset index for clean DataFrame
)

# Clean up the product category names for better readability
# Replace underscores with spaces and capitalize each word
data_products["product_category"] = data_products["product_category_name"].apply(
    lambda x: str(x).replace("_", " ").title()
)

data_products.head()

,product_category_name,product_counts,total_price,product_category
0,cama_mesa_banho,11115,1036988.68,Cama Mesa Banho
1,beleza_saude,9670,1258681.34,Beleza Saude
2,esporte_lazer,8641,988048.97,Esporte Lazer
3,moveis_decoracao,8334,729762.49,Moveis Decoracao
4,informatica_acessorios,7827,911954.32,Informatica Acessorios


### Top Products Sold By Category

In [230]:
fig = go.Figure()

fig.add_trace(
    go.Bar (
        x = data_products["product_category"].head(15),
        y = data_products["product_counts"].head(15),
        name = "Number of products sold",
        marker_color = "rgb(64,224,208)"
    )
)

fig.add_trace(
    go.Scatter (
        x = data_products["product_category"].head(15),
        y = data_products["total_price"].head(15),
        name = "GMV",
        yaxis = "y2",
        marker_color = "rgb(255,160,122)",
        mode="lines+markers"
    )
)

fig.update_layout(
    title = dict(text="<b>Top 15 Selling Products Categories<b>",
                 font=dict(size=12, family="Arial", color="black")),
    plot_bgcolor = "white",
    yaxis = dict(side = "left", showgrid = False, zeroline = True, showline = False, showticklabels = False),
    yaxis2 = dict(side = "right", overlaying = "y", showgrid = False, zeroline = False, showline = False, showticklabels = False),
    xaxis = dict(showline = True, linecolor = "rgb(204, 204, 204)", linewidth = 2),
    legend = dict(orientation="h", x=0.8, y = 1.1),
    annotations=[dict(text="Created By Thuan Dao.", xref="paper", yref="paper", x=1, y=-.3,
    showarrow=False, font=dict(size=10, color="gray", family="Arial"))],
    hovermode = "x unified"
)


<font color = '#e3f56e'>
Insight

* **Garden Care Tools** lead in both sales volume and GMV → core product category.
* **Beauty Supplies**, **Home Decor**, and **Vehicle Accessories** also have high revenue → strong consumer demand.
* Some categories like **Phone Accessories** and **PC Accessories** have high sales volume but low GMV → low average order value.
* Conversely, **Furniture Accessories** and **Vehicle Electronics** have high GMV relative to sales volume → high-value products.
* Differences between volume and GMV reflect varying consumer behavior across categories.

</font>

### Products Segmentation

In [231]:
# Aggregate product sales data for delivered orders
data_products_seg = (
    # Step 1: Merge order items with orders, keeping only delivered orders
    df_order_items.merge(
        df_orders[["order_id", "order_status"]].query('order_status == "delivered"'),
        how="inner",
        on="order_id"
    )
    # Step 2: Group by product_id to calculate sales metrics
    .groupby("product_id")
    .agg(
        sales_volume=("product_id", "count"),  # number of items sold
        sales_values=("price", "sum")          # total sales value
    )
    # Step 3: Sort products by total sales value in descending order
    .sort_values(by="sales_values", ascending=False)
    .reset_index()  # reset index for clean DataFrame
)

display(data_products_seg.head())

,product_id,sales_volume,sales_values
0,bb50f2e236e5eea0100680137654686c,194,63560.00
1,6cdd53843498f92890544667809f1595,153,53652.30
2,d6160fb7873f184099d9bc95e30376af,33,45949.35
3,d1c427060a0f73f6b889a5c7c61f2ac4,332,45620.56
4,99a4788cb24856965c36a24e339b6058,477,42049.66


In [232]:
# Create a historical sales dataset for delivered orders
data_sales_his = (
    # Step 1: Filter only delivered orders and select relevant columns
    df_orders.query('order_status == "delivered"')[["order_id", "order_purchase_timestamp"]]
    # Step 2: Merge with order items to get product-level details
    .merge(df_order_items[["order_id", "product_id"]], how="left", on="order_id")
)

# Extract only the date part from the timestamp for aggregation
data_sales_his["order_purchase_date"] = data_sales_his["order_purchase_timestamp"].dt.date

# Aggregate daily product sales
data_sales_his = (
    data_sales_his[["product_id", "order_purchase_date"]]
    .groupby(["order_purchase_date", "product_id"])
    .agg(
        product_sold=("product_id", "count")  # count number of products sold per day
    )
    .reset_index()  # reset index for clean DataFrame
)

display(data_sales_his.head())

,order_purchase_date,product_id,product_sold
0,2016-09-15,5a6b04657a4c5ee34285d1e4619a96b4,3
1,2016-10-03,107177bf61755f05c604fe57e02467d6,1
2,2016-10-03,3ae08df6bcbfe23586dd431c40bddbb7,1
3,2016-10-03,a5c3ddb1a400f50d1cf7138727aec136,1
4,2016-10-03,b72b39418216e944bb34e35f4d3ea8c7,1


In [233]:
# Rank each purchase date sequentially to create a "day" index
data_sales_his["day"] = data_sales_his["order_purchase_date"].rank(method="dense", ascending=True).astype(int)

# Pivot the DataFrame to have one row per product and one column per day
# Values are number of products sold on that day
data_sales_his = (
    data_sales_his
    .pivot(index="product_id", columns="day", values="product_sold")
    .fillna(0)  # fill missing sales with 0
    .astype(int)
    .reset_index()
)

# Rename day columns to "day_1", "day_2", ...
data_sales_his.columns = [data_sales_his.columns[0]] + ["day_" + str(col) for col in data_sales_his.columns[1:]]

# Store the day columns for later calculations
sales_col = data_sales_his.columns[1:]

# Merge product-level sales summary (e.g., sales volume and value)
data_sales_his = data_sales_his.merge(data_products_seg, how="left", on="product_id")

# Calculate the mean daily sales for each product
data_sales_his["mean"] = data_sales_his[sales_col].mean(axis=1)

# Keep only products with at least some sales
data_sales_his = data_sales_his.query("mean > 0")

# Calculate the standard deviation of daily sales for each product
data_sales_his["std"] = data_sales_his[sales_col].std(axis=1)

# Calculate the coefficient of variation (std / mean) for each product
# This measures sales variability relative to mean sales
data_sales_his["coef"] = data_sales_his["std"] / data_sales_his["mean"]

display(data_sales_his.head())

,product_id,day_1,day_2,day_3,day_4,day_5,day_6,day_7,day_8,day_9,day_10,day_11,day_12,day_13,day_14,day_15,day_16,day_17,day_18,day_19,day_20,day_21,day_22,day_23,day_24,day_25,day_26,day_27,day_28,day_29,day_30,day_31,day_32,day_33,day_34,day_35,day_36,day_37,day_38,day_39,day_40,day_41,day_42,day_43,day_44,day_45,day_46,day_47,day_48,day_49,day_50,day_51,day_52,day_53,day_54,day_55,day_56,day_57,day_58,day_59,day_60,day_61,day_62,day_63,day_64,day_65,day_66,day_67,day_68,day_69,day_70,day_71,day_72,day_73,day_74,day_75,day_76,day_77,day_78,day_79,day_80,day_81,day_82,day_83,day_84,day_85,day_86,day_87,day_88,day_89,day_90,day_91,day_92,day_93,day_94,day_95,day_96,day_97,day_98,day_99,day_100,day_101,day_102,day_103,day_104,day_105,day_106,day_107,day_108,day_109,day_110,day_111,day_112,day_113,day_114,day_115,day_116,day_117,day_118,day_119,day_120,day_121,day_122,day_123,day_124,day_125,day_126,day_127,day_128,day_129,day_130,day_131,day_132,day_133,day_134,day_135,day_136,day_137,day_138,day_139,day_140,day_141,day_142,day_143,day_144,day_145,day_146,day_147,day_148,day_149,day_150,day_151,day_152,day_153,day_154,day_155,day_156,day_157,day_158,day_159,day_160,day_161,day_162,day_163,day_164,day_165,day_166,day_167,day_168,day_169,day_170,day_171,day_172,day_173,day_174,day_175,day_176,day_177,day_178,day_179,day_180,day_181,day_182,day_183,day_184,day_185,day_186,day_187,day_188,day_189,day_190,day_191,day_192,day_193,day_194,day_195,day_196,day_197,day_198,day_199,day_200,day_201,day_202,day_203,day_204,day_205,day_206,day_207,day_208,day_209,day_210,day_211,day_212,day_213,day_214,day_215,day_216,day_217,day_218,day_219,day_220,day_221,day_222,day_223,day_224,day_225,day_226,day_227,day_228,day_229,day_230,day_231,day_232,day_233,day_234,day_235,day_236,day_237,day_238,day_239,day_240,day_241,day_242,day_243,day_244,day_245,day_246,day_247,day_248,day_249,...,day_368,day_369,day_370,day_371,day_372,day_373,day_374,day_375,day_376,day_377,day_378,day_379,day_380,day_381,day_382,day_383,day_384,day_385,day_386,day_387,day_388,day_389,day_390,day_391,day_392,day_393,day_394,day_395,day_396,day_397,day_398,day_399,day_400,day_401,day_402,day_403,day_404,day_405,day_406,day_407,day_408,day_409,day_410,day_411,day_412,day_413,day_414,day_415,day_416,day_417,day_418,day_419,day_420,day_421,day_422,day_423,day_424,day_425,day_426,day_427,day_428,day_429,day_430,day_431,day_432,day_433,day_434,day_435,day_436,day_437,day_438,day_439,day_440,day_441,day_442,day_443,day_444,day_445,day_446,day_447,day_448,day_449,day_450,day_451,day_452,day_453,day_454,day_455,day_456,day_457,day_458,day_459,day_460,day_461,day_462,day_463,day_464,day_465,day_466,day_467,day_468,day_469,day_470,day_471,day_472,day_473,day_474,day_475,day_476,day_477,day_478,day_479,day_480,day_481,day_482,day_483,day_484,day_485,day_486,day_487,day_488,day_489,day_490,day_491,day_492,day_493,day_494,day_495,day_496,day_497,day_498,day_499,day_500,day_501,day_502,day_503,day_504,day_505,day_506,day_507,day_508,day_509,day_510,day_511,day_512,day_513,day_514,day_515,day_516,day_517,day_518,day_519,day_520,day_521,day_522,day_523,day_524,day_525,day_526,day_527,day_528,day_529,day_530,day_531,day_532,day_533,day_534,day_535,day_536,day_537,day_538,day_539,day_540,day_541,day_542,day_543,day_544,day_545,day_546,day_547,day_548,day_549,day_550,day_551,day_552,day_553,day_554,day_555,day_556,day_557,day_558,day_559,day_560,day_561,day_562,day_563,day_564,day_565,day_566,day_567,day_568,day_569,day_570,day_571,day_572,day_573,day_574,day_575,day_576,day_577,day_578,day_579,day_580,day_581,day_582,day_583,day_584,day_585,day_586,day_587,day_588,day_589,day_590,day_591,day_592,day_593,day_594,day_595,day_596,day_597,day_598,day_599,day_600,day_601,day_602,day_603,day_604,day_605,day_606,day_607,day_608,day_609,day_610,day_611,day_612,sales_volume,sales_values,mean,std,coef
0,00066f42aeeb9f3007548bb9d3f33c38,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [234]:
# Create a summary of product-level sales by dropping daily sales columns
data_product_summary = data_sales_his.copy().drop(columns=sales_col, axis=1)

# Calculate total sales value across all products
total_sales_values = data_product_summary["sales_values"].sum()

# Calculate percentage contribution of each product to total sales
data_product_summary["%sales"] = data_product_summary["sales_values"] / total_sales_values * 100

# Sort products by percentage sales contribution in descending order
data_product_summary.sort_values(by="%sales", ascending=False, inplace=True, ignore_index=True)

# Calculate cumulative sales percentage (useful for Pareto/ABC analysis)
data_product_summary["%sales_cs"] = data_product_summary["%sales"].cumsum()

display(data_product_summary.head())

,product_id,sales_volume,sales_values,mean,std,coef,%sales,%sales_cs
0,bb50f2e236e5eea0100680137654686c,194,63560.00,0.316993,0.728767,2.298998,0.480732,0.480732
1,6cdd53843498f92890544667809f1595,153,53652.30,0.250000,0.614540,2.458161,0.405796,0.886528
2,d6160fb7873f184099d9bc95e30376af,33,45949.35,0.053922,0.331689,6.151316,0.347535,1.234063
3,d1c427060a0f73f6b889a5c7c61f2ac4,332,45620.56,0.542484,0.987562,1.820446,0.345048,1.579112
4,99a4788cb24856965c36a24e339b6058,477,42049.66,0.779412,1.211795,1.554755,0.318040,1.897152


In [235]:
# Total number of products
n_products = len(data_product_summary)

# Define the number of products in each ABC category
n_a, n_b = int(0.05 * n_products), int(0.2 * n_products)  # top 5% = A, next 15% = B, rest = C

# Assign a unique SKU ID to each product
data_product_summary["sku_id"] = pd.Series(range(1, n_products + 1)).astype(int)

# Assign ABC categories based on product ranking
# - "A": top 5% of products by sales contribution
# - "B": next 15% of products
# - "C": remaining products
data_product_summary["abc"] = pd.Series(range(n_products)).apply(
    lambda x: "A" if x <= n_a - 1 else "B" if x <= n_b - 1 else "C"
)

display(data_product_summary.head())

,product_id,sales_volume,sales_values,mean,std,coef,%sales,%sales_cs,sku_id,abc
0,bb50f2e236e5eea0100680137654686c,194,63560.00,0.316993,0.728767,2.298998,0.480732,0.480732,1,A
1,6cdd53843498f92890544667809f1595,153,53652.30,0.250000,0.614540,2.458161,0.405796,0.886528,2,A
2,d6160fb7873f184099d9bc95e30376af,33,45949.35,0.053922,0.331689,6.151316,0.347535,1.234063,3,A
3,d1c427060a0f73f6b889a5c7c61f2ac4,332,45620.56,0.542484,0.987562,1.820446,0.345048,1.579112,4,A
4,99a4788cb24856965c36a24e339b6058,477,42049.66,0.779412,1.211795,1.554755,0.318040,1.897152,5,A


In [236]:
# Loop through each ABC product class and summarize its contribution
for product_class in ["A", "B", "C"]:
    # Filter products belonging to the current class
    filter = data_product_summary[data_product_summary["abc"] == product_class]
    
    # Calculate the percentage of products in this class
    percentage = len(filter) / len(data_product_summary) * 100
    
    # Get the highest SKU ID in this class (as a proxy for number of SKUs)
    sku_number = filter["sku_id"].max()
    
    # Calculate total sales percentage contributed by this class
    sales_percent = filter["%sales"].sum()
    
    # Print a summary of the class
    print(f"Class {product_class} ({percentage:.0f}%) has {sku_number} SKUs and {sales_percent:.2f}% of turnover")

Class A (5%) has 1610 SKUs and 47.82% of turnover
Class B (15%) has 6443 SKUs and 27.00% of turnover
Class C (80%) has 32216 SKUs and 25.18% of turnover


In [237]:
import math

data_product_plot = data_product_summary.copy()
class_mapping = {"A": "mediumturquoise", "B": "indianred", "C": "limegreen"}
fig = go.Figure()

for prod_class in class_mapping.keys():
    fig.add_trace(
        go.Scatter(
            x=data_product_plot[data_product_plot["abc"] == prod_class]["%sales"],
            y=data_product_plot[data_product_plot["abc"] == prod_class]["coef"],
            name = "Product Class " + str(prod_class),
            marker=dict(color=class_mapping[prod_class]),
            mode="markers"
        )
    )
fig.update_layout(
    title = dict(text="<b>Distribution by Demand Variability<b>",
                 font=dict(size=12, family="Arial", color="black")),
    plot_bgcolor = "white",
    yaxis = dict(side = "left", linecolor = "rgb(204, 204, 204)", showline = True, linewidth = 2, title = "Variability of the demand",
                range = [math.floor(min(data_product_summary["coef"])), math.ceil(max(data_product_summary["coef"])) + 1]),
    xaxis = dict(showline = True, linecolor = "rgb(204, 204, 204)", linewidth = 2, title = "Percentage of Turnover (%)",
            range = [math.floor(min(data_product_summary["%sales"])) - 0.01, (max(data_product_summary["%sales"])) + 0.01]),
    legend = dict(orientation="h"),
    annotations=[dict(text="Created By Thuan Dao.", xref="paper", yref="paper", x=1, y=-.2,
    showarrow=False, font=dict(size=10, color="gray", family="Arial"))]
)

In [238]:
fig = make_subplots(
    rows=1, cols=3,
    start_cell="top-left",
    subplot_titles=("Class A", "Class B", "Class C")
)

product_classes = ["A", "B", "C"]
col = 1

for product_class in product_classes:
    x = data_product_summary[data_product_summary["abc"] == product_class]["sales_values"]
    trace = go.Histogram(
        x=x,
        nbinsx=50,
        marker=dict(color="mediumturquoise" if product_class == "A"
                    else "indianred" if product_class == "B"
                    else "limegreen")
    )
    fig.append_trace(trace, 1, col)
    fig.update_xaxes(
        title_text="Sales Values",
        title_font=dict(size=10, family="Arial", color="black"),
        row=1, col=col
    )
    fig.update_yaxes(
        title_text="Product Count",
        title_font=dict(size=10, family="Arial", color="black"),
        row=1, col=col
    )
    col += 1

fig.update_layout(
    title=dict(
        text="<b>Sales Distribution By Product Class<b>",
        font=dict(size=12, family="Arial", color="black")
    ),
    plot_bgcolor="white",
    showlegend=False,
    bargap=0.01
)

fig.add_annotation(
    text="Created By Thuan Dao.",
    xref="paper", yref="paper",
    x=1, y=-0.2,
    showarrow=False,
    font=dict(size=10, color="gray", family="Arial")
)

fig.show()

<font color = '#e3f56e'>
Insight

* **Class A** → high-value products (up to 60K) but mostly low sales → right-skewed, few top sellers drive revenue.
* **Class B** → moderate, evenly distributed sales (0–1.4K) → stable category.
* **Class C** → low-value products (max ~400) but many items → high frequency, low order value.
</font>

In [239]:
# Calculate a volatility threshold as the midpoint of the range of standard deviations
volatility_threshold = (max(data_product_summary["std"]) - min(data_product_summary["std"])) / 2

# Define a function to classify products based on sales percentage and volatility
def classify(product):
    if product["%sales"] <= 0.25 and product["std"] <= volatility_threshold:
        return "Low Volume, Low Volatility"
    elif product["%sales"] > 0.25 and product["std"] <= volatility_threshold:
        return "High Volume, Low Volatility"
    elif product["%sales"] <= 0.25 and product["std"] > volatility_threshold:
        return "Low Volume, High Volatility"
    else:
        return "High Volume, High Volatility"

# Apply the classification function to each product
data_product_summary["type"] = data_product_summary.apply(lambda x: classify(x), axis=1)

display(data_product_summary.head())

,product_id,sales_volume,sales_values,mean,std,coef,%sales,%sales_cs,sku_id,abc,type
0,bb50f2e236e5eea0100680137654686c,194,63560.00,0.316993,0.728767,2.298998,0.480732,0.480732,1,A,"High Volume, Low Volatility"
1,6cdd53843498f92890544667809f1595,153,53652.30,0.250000,0.614540,2.458161,0.405796,0.886528,2,A,"High Volume, Low Volatility"
2,d6160fb7873f184099d9bc95e30376af,33,45949.35,0.053922,0.331689,6.151316,0.347535,1.234063,3,A,"High Volume, Low Volatility"
3,d1c427060a0f73f6b889a5c7c61f2ac4,332,45620.56,0.542484,0.987562,1.820446,0.345048,1.579112,4,A,"High Volume, High Volatility"
4,99a4788cb24856965c36a24e339b6058,477,42049.66,0.779412,1.211795,1.554755,0.318040,1.897152,5,A,"High Volume, High Volatility"


In [240]:
type_mapping = {
    "Low Volume, Low Volatility": {
        "info" : "Easy/ Low ROI",
        "color": "powderblue",
        "x": [0, 2, 2, 0],
        "y": [0, 0, 2, 2]
    },
    "Low Volume, High Volatility": {
        "info" : "Difficult/ Low ROI",
        "color": "orangered",
        "x": [0, 0, 2, 2],
        "y": [2, 4, 4, 2],
    },
    "High Volume, Low Volatility": {
        "info" : "Moderate+/ High ROI",
        "color": "skyblue",
        "x": [2, 4, 4, 2],
        "y": [0, 0, 2, 2],
    },
    "High Volume, High Volatility": {
        "info" : "Critical+/ Moderate+/ High ROI",
        "color": "yellowgreen",
        "x": [2, 4, 4, 2],
        "y": [2, 2, 4, 4],
    }
}
def plot (x, y, _type_):
    return go.Scatter (
        x = x,
        y = y,
        fill = "toself",
        fillcolor = type_mapping[_type_]["color"],
        hoveron = "fills",
        hoverinfo = "text",
        line_color = "white",
        mode = "lines+text",
        name = _type_,
    )
total_values = data_product_summary["sales_values"].sum()
def plot_annotation (x, y, _type_):
    total_products_sold = data_product_summary[data_product_summary["type"] == _type_]["product_id"].nunique()
    total_sales_volume = data_product_summary[data_product_summary["type"] == _type_]["sales_volume"].sum()
    total_sales_values = data_product_summary[data_product_summary["type"] == _type_]["sales_values"].sum()
    values_percent = total_sales_values/total_values * 100
    info = type_mapping[_type_]["info"]
    text = f"<b>{_type_}</b><br>{info}<br>Total Products Sold: {total_products_sold}<br>Total Sales Volume: {total_sales_volume}<br>Total Sales Value: {total_sales_values:.2f} ({values_percent:.2f}%)"
    return fig.add_annotation (
        x = x, y = y , font = dict(color = "black", size = 12),text = text, align= "left", xanchor = "left", showarrow = False
    )

In [241]:
fig = go.Figure()
for _type_ in set(data_product_summary["type"]):
    x = type_mapping[_type_]["x"]
    y = type_mapping[_type_]["y"]
    fig.add_trace (
        plot (x, y, _type_)
    )
    plot_annotation (x[0] + 0.05, y[0] + 0.7, _type_)

fig.update_layout(
    title=dict(
        text="<b>Sales Distribution By Product Class<b>",
        font=dict(size=12, family="Arial", color="black")
    ),
    plot_bgcolor = "white",
    xaxis = dict(showline = False,  range = [0, 4],  rangemode = "nonnegative"),
    yaxis = dict(showline = False, range = [0, 4], rangemode = "nonnegative",  tickmode="array", tickvals=[1, 2, 3, 4]),
    showlegend = False
)

fig.add_annotation(
    text="Created By Thuan Dao.",
    xref="paper", yref="paper",
    x=1, y=-0.2,
    showarrow=False,
    font=dict(size=10, color="gray", family="Arial")
)
fig.show()

## E-commerce Operation Analysis

### Time-series Analysis

#### Daily GMV

In [242]:
# Merge orders with order items to get product-level details and filter only delivered orders
data_orders_timeseries_daily = df_orders.merge(df_order_items_, how="left", on="order_id")\
                                        .query('order_status == "delivered"')

# Extract only the date part from the delivered date for daily aggregation
data_orders_timeseries_daily["date"] = data_orders_timeseries_daily["order_delivered_customer_date"].dt.to_period("D")

# Aggregate daily metrics: number of orders, total product counts, total price
data_orders_timeseries_daily = (
    data_orders_timeseries_daily
    .groupby("date")
    .agg({
        "order_id": "count",          # daily order count
        "product_counts": "sum",      # daily total products sold
        "total_price": "sum",         # daily total sales value
    })
    .query("date.isna() == False")   # remove any rows with missing dates
    .reset_index()
)

# Calculate 7-day moving average of total sales value
data_orders_timeseries_daily["7d_moving_average"] = (
    data_orders_timeseries_daily.sort_values(by="date")[["total_price"]]
    .transform(lambda x: round(x.rolling(7).mean(), 2))
)

# Calculate 30-day moving average of total sales value
data_orders_timeseries_daily["30d_moving_average"] = (
    data_orders_timeseries_daily.sort_values(by="date")[["total_price"]]
    .transform(lambda x: round(x.rolling(30).mean(), 2))
)

In [243]:
fig = go.Figure()

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries_daily["date"].astype(str),
        y = data_orders_timeseries_daily["total_price"],
        mode = "markers",
        name = "GMV",
        marker_color = "#da3644",
    )
)

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries_daily["date"].astype(str),
        y = data_orders_timeseries_daily["7d_moving_average"],
        mode = "lines",
        name = "7-D Moving Average",
        marker_color = "#03b6fc",
    )
)

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries_daily["date"].astype(str),
        y = data_orders_timeseries_daily["30d_moving_average"],
        mode = "lines",
        name = "30-D Moving Average",
        marker_color = "#1de02d",
    )
)

fig.update_layout (
    title=dict(
        text="<b>Daily Revenue<b>",
        font=dict(size=12, family="Arial", color="black")
    ),
    plot_bgcolor = "white",
    xaxis = dict (
        showline = True,
        linecolor = "rgb(204, 204, 204)",
        linewidth = 1.5,
    ),
    yaxis = dict (showticklabels = True),
    legend = dict(orientation = "h"),
    hovermode = "x unified",
    showlegend=True
)

fig.add_annotation(
    text="Created By Thuan Dao.",
    xref="paper", yref="paper",
    x=1, y=-0.2,
    showarrow=False,
    font=dict(size=10, color="gray", family="Arial")
)
fig.show()

<font color = '#e3f56e'>
Insight

- The daily GMV value flactuated over the period, which can range from 0 to over 60.000. However, the overall trend was climbing up. <br>
- One of abnormal point here is from Oct 2016 to Jan 2017, and from Oct 2018, the GMV seems to flatten down before 5.000.
</font>

#### Monthly GMV

In [244]:
# Merge orders with order items to get product-level details and filter only delivered orders
data_orders_timeseries_monthly = df_orders.merge(df_order_items_, how="left", on="order_id")\
                                         .query('order_status == "delivered"')

# Extract year-month period from the delivered date for monthly aggregation
data_orders_timeseries_monthly["year_month"] = data_orders_timeseries_monthly["order_delivered_customer_date"].dt.to_period("M")

# Aggregate monthly metrics: number of orders, total products, total sales
data_orders_timeseries_monthly = (
    data_orders_timeseries_monthly
    .groupby("year_month")
    .agg({
        "order_id": "count",          # monthly order count
        "product_counts": "sum",      # monthly total products sold
        "total_price": "sum",         # monthly total sales value
    })
    .query("year_month.isna() == False")  # remove rows with missing dates
    .rename(columns={"order_id": "orders_count"})  # rename for clarity
    .reset_index()
)

# Calculate 3-month moving average of total sales value
data_orders_timeseries_monthly["moving_average"] = (
    data_orders_timeseries_monthly.sort_values(by="year_month")[["total_price"]]
    .transform(lambda x: round(x.rolling(3).mean(), 2))
    .fillna(0)  # fill NaN values for first 2 months
)

# Display the first 5 rows of monthly time series with moving average
display(data_orders_timeseries_monthly.head())

,year_month,orders_count,product_counts,total_price,moving_average
0,2016-10,216,241.0,29874.44,0.00
1,2016-11,61,72.0,9837.68,0.00
2,2016-12,4,4.0,758.86,13490.33
3,2017-01,294,326.0,33599.12,14731.89
4,2017-02,1410,1565.0,198909.29,77755.76


In [245]:
fig = go.Figure()

fig.add_trace(
    go.Bar (
        x = data_orders_timeseries_monthly["year_month"].astype(str),
        y = data_orders_timeseries_monthly["orders_count"],
        name = "Number of orders",
        marker_color = "rgb(64,224,208)",
    )
)

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries_monthly["year_month"].astype(str),
        y = data_orders_timeseries_monthly["total_price"],
        name = "GMV",
        yaxis = "y2",
        marker_color = "rgb(255,160,122)",
        mode="lines+markers"
    )
)

fig.update_layout (
    title=dict(
        text="<b>Monthly GMV<b>",
        font=dict(size=12, family="Arial", color="black")
    ),
    plot_bgcolor = "white",
    xaxis = dict (
        showline = True,
        showgrid = False,
        linecolor = "rgb(204, 204, 204)",
        linewidth = 1.5,
    ),
    yaxis = dict (
        side = "left",
        showgrid = False,
        zeroline = True,
        showline = False,
        showticklabels = False,
    ),
    yaxis2 = dict (
        side = "right",
        overlaying = "y",
        showgrid = False,
        zeroline = False,
        showline = False,
        showticklabels = False,
    ),
    legend = dict (
        orientation="h",
    ),
    hovermode = "x unified"
)

fig.add_annotation(
    text="Created By Thuan Dao.",
    xref="paper", yref="paper",
    x=1, y=-0.2,
    showarrow=False,
    font=dict(size=10, color="gray", family="Arial")
)
fig.show()

<font color = '#e3f56e'>
Insight

* **Growth trend**: GMV and order count steadily increased from late 2016 to mid-2018 → stable business growth.
* **Peak**: GMV peaked around July 2018 → most effective sales period.
* **Sharp decline**: GMV and orders dropped sharply in Oct 2018, with GMV falling more → investigate causes (market, products, operations, etc.).

</font>

#### Sales Prediction Model

In [246]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# Work with a copy of the monthly orders time series data
data_orders_timeseries = data_orders_timeseries_monthly.copy()

# Calculate month-over-month difference in total sales
data_orders_timeseries["total_price_diff"] = data_orders_timeseries["total_price"].diff()

# Add a month index (1, 2, 3, ...) to represent time
data_orders_timeseries["month"] = data_orders_timeseries.index + 1

# Prepare features for time series regression
# Start with the difference in sales (diff) column
sales_data = data_orders_timeseries[["total_price_diff"]].copy(deep=True).fillna(0)

# Store actual total sales for later evaluation
sales_actual = data_orders_timeseries["total_price"].to_list()

# Create lag features for the past 12 months
for month in range(1, 13):
    col_name = "month_" + str(month)
    sales_data[col_name] = sales_data["total_price_diff"].shift(month)  # lagged difference

# Drop rows with NaN values created by lagging and reset index
sales_data = sales_data.dropna().reset_index(drop=True)

display(sales_data.head())

,total_price_diff,month_1,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,77296.91,44738.43,80528.94,24221.41,-89361.87,251454.22,-67198.98,136976.92,165310.17,32840.26,-9078.82,-20036.76,0.00
1,-6107.75,77296.91,44738.43,80528.94,24221.41,-89361.87,251454.22,-67198.98,136976.92,165310.17,32840.26,-9078.82,-20036.76
2,293535.38,-6107.75,77296.91,44738.43,80528.94,24221.41,-89361.87,251454.22,-67198.98,136976.92,165310.17,32840.26,-9078.82
3,-93874.78,293535.38,-6107.75,77296.91,44738.43,80528.94,24221.41,-89361.87,251454.22,-67198.98,136976.92,165310.17,32840.26
4,-101147.56,-93874.78,293535.38,-6107.75,77296.91,44738.43,80528.94,24221.41,-89361.87,251454.22,-67198.98,136976.92,165310.17


In [247]:
# Split the data into training (first 10 months) and test set (remaining months)
train_data = sales_data[:10]
test_data = sales_data[10:]

# Scale features to range [-1, 1] using MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train_data)

# Apply scaling to train and test sets
train_data = scaler.transform(train_data)
test_data = scaler.transform(test_data)

# Separate features (lagged months) and target (current month diff)
X_train, y_train = train_data[:, 1:], train_data[:, 0:1].ravel()  # target is first column
X_test, y_test = test_data[:, 1:], test_data[:, 0:1].ravel()

# Define a regression pipeline using spline transformation + Ridge regression
model = make_pipeline(
    SplineTransformer(
        knots=np.linspace(0, np.pi**2 + 5, 36).reshape(3, 12),  # spline knots
        degree=2,                                               # quadratic spline
        extrapolation="periodic"                                # handle out-of-range data periodically
    ),
    Ridge(alpha=1e-8, max_iter=10000, random_state=42)         # Ridge regression
)

# Fit the model on training data
model.fit(X_train, y_train)

# Predict differences for test set
y_pred = model.predict(X_test).reshape(-1, 1)

# Combine predictions with lagged features for inverse scaling
y_pred = np.concatenate([y_pred, X_test], axis=1)

# Inverse transform to original scale
sales_pred = scaler.inverse_transform(y_pred)

# Add predicted difference to actual previous sales to get forecasted total sales
result = []
for index in range(len(sales_pred)):
    if (sales_pred[index][0] + sales_actual[index]) < 0:
        result.append(0)  # avoid negative sales
    else:
        result.append(sales_pred[index][0] + sales_actual[index])

# Calculate R^2 score for the last 3 months
accuracy = r2_score(sales_actual[-3:], result)
print(f"The accuracy is {accuracy*100:.2f}%")

The accuracy is 72.91%


In [248]:
fig = go.Figure()
fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries["year_month"].astype(str),
        y = data_orders_timeseries["total_price"],
        name = "Actual Sales",
        marker_color = "rgb(255,160,122)",
        mode = "lines+markers"
    )
)

fig.add_trace(
    go.Scatter (
        x = data_orders_timeseries["year_month"].astype(str).to_list()[-3:],
        y = result,
        name = "Predicted Sales",
        marker_color = "lightseagreen",
        mode = "lines+markers"
    )
)

fig.update_layout (
    title=dict(
        text="<b>Actual vs Predicted GMV<b>",
        font=dict(size=12, family="Arial", color="black")
    ),
    plot_bgcolor = "white",
    xaxis = dict (
        showline = True,
        showgrid = False,
        linecolor = "rgb(204, 204, 204)",
        linewidth = 1.5,
    ),
    legend = dict (
        orientation="h",
    ),
    hovermode = "x unified"
)

fig.add_annotation(
    text="Created By Thuan Dao.",
    xref="paper", yref="paper",
    x=1, y=-0.2,
    showarrow=False,
    font=dict(size=10, color="gray", family="Arial")
)
fig.show()

<font color = '#e3f56e'>
Insight

* **Actual GMV** grew steadily from late 2016 to mid-2018, peaking in July 2018, then dropped sharply.
* **Predicted GMV** appears only from July to October 2018, showing large deviations, especially at the peak.
* Prediction model fails to capture real trends → needs improvement in accuracy and responsiveness to market changes.

</font>

### Customers' Satisfaction

In [249]:
# Aggregate review scores at the order level
data_review = (
    df_order_reviews[["order_id", "review_score"]]  # select relevant columns
    .groupby("order_id")                            # group by each order
    .agg({"review_score": "mean"})                 # calculate average review score per order
    .reset_index()                                  # reset index for clean DataFrame
)

display(data_review.head())

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [250]:
# Create a comprehensive orders dataset by merging multiple sources
data_orders_ = (
    df_orders.copy(deep=True)  # make a deep copy of the orders data
    # Merge average review score per order
    .merge(data_review, how="left", on="order_id")
    # Merge order items info: seller_id and freight_value
    .merge(df_order_items[["order_id", "seller_id", "freight_value"]], how="left", on="order_id")
    # Merge seller info: city and state
    .merge(df_sellers[["seller_id", "seller_city", "seller_state"]], how="left", on="seller_id")
    # Merge customer info: city and state
    .merge(df_customers[["customer_id", "customer_city", "customer_state"]], how="left", on="customer_id")
    # Remove duplicate rows keeping the first occurrence
    .drop_duplicates(keep="first")
    # Reset index for clean DataFrame
    .reset_index(drop=True)
)

# Fill missing review scores with 0 (for orders with no reviews)
data_orders_["review_score"].fillna(0, inplace=True)

display(data_orders_.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,seller_id,freight_value,seller_city,seller_state,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,4.0,3504c0cb71d7fa48d967e0e4c94d59d9,8.72,maua,SP,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,4.0,289cdb325fb7e7f891c38608bf9e0962,22.76,belo horizonte,SP,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,5.0,4869f7a5dfa277a7dca6462dcf3b52b2,19.22,guariba,SP,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,5.0,66922902710d126a0e7d26b0e3805106,27.20,belo horizonte,MG,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,5.0,2c9e548be18521d1c43cde1c582c6de8,8.72,mogi das cruzes,SP,santo andre,SP


In [251]:
# Extract year-month from order purchase timestamp for monthly analysis
data_orders_["year_month"] = data_orders_["order_purchase_timestamp"].dt.to_period("M")

# Calculate the time (in minutes) from order purchase to approval
data_orders_["approved_mins"] = (
    (data_orders_["order_approved_at"] - data_orders_["order_purchase_timestamp"])
    .dt.total_seconds() / 60
).fillna(0).astype(int)

# Calculate the time (in days) from order approval to carrier delivery
data_orders_["to_carrier_days"] = (
    (data_orders_["order_delivered_carrier_date"] - data_orders_["order_approved_at"])
    .dt.days
).fillna(0).astype(int)

# Calculate the time (in days) from carrier delivery to customer delivery
data_orders_["to_customer_days"] = (
    (data_orders_["order_delivered_customer_date"] - data_orders_["order_delivered_carrier_date"])
    .dt.days
).fillna(0).astype(int)

# Total delivery time (from approval to customer delivery)
data_orders_["delivery_days"] = data_orders_["to_carrier_days"] + data_orders_["to_customer_days"]

# Flag orders that were delivered later than the estimated delivery date
data_orders_["is_late"] = np.where(
    data_orders_["order_estimated_delivery_date"] <= data_orders_["order_delivered_customer_date"], 
    True, False)

display(data_orders_.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,seller_id,freight_value,seller_city,seller_state,customer_city,customer_state,year_month,approved_mins,to_carrier_days,to_customer_days,delivery_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,4.0,3504c0cb71d7fa48d967e0e4c94d59d9,8.72,maua,SP,sao paulo,SP,2017-10,10,2,6,8,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,4.0,289cdb325fb7e7f891c38608bf9e0962,22.76,belo horizonte,SP,barreiras,BA,2018-07,1842,0,12,12,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,5.0,4869f7a5dfa277a7dca6462dcf3b52b2,19.22,guariba,SP,vianopolis,GO,2018-08,16,0,9,9,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,5.0,66922902710d126a0e7d26b0e3805106,27.20,belo horizonte,MG,sao goncalo do amarante,RN,2017-11,17,3,9,12,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,5.0,2c9e548be18521d1c43cde1c582c6de8,8.72,mogi das cruzes,SP,santo andre,SP,2018-02,61,0,1,1,False


In [252]:
columns_list = ["order_id", "customer_id", "customer_city", "customer_state", "seller_id", "seller_city",
       "seller_state", "order_status", "order_purchase_timestamp",
       "review_score",  "freight_value",  "year_month",
       "approved_mins", "to_carrier_days", "to_customer_days", "delivery_days", "is_late"]

data_orders_ = data_orders_[columns_list]
display(data_orders_.head())

,order_id,customer_id,customer_city,customer_state,seller_id,seller_city,seller_state,order_status,order_purchase_timestamp,review_score,freight_value,year_month,approved_mins,to_carrier_days,to_customer_days,delivery_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,sao paulo,SP,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP,delivered,2017-10-02 10:56:33,4.0,8.72,2017-10,10,2,6,8,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,BA,289cdb325fb7e7f891c38608bf9e0962,belo horizonte,SP,delivered,2018-07-24 20:41:37,4.0,22.76,2018-07,1842,0,12,12,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,vianopolis,GO,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,delivered,2018-08-08 08:38:49,5.0,19.22,2018-08,16,0,9,9,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,RN,66922902710d126a0e7d26b0e3805106,belo horizonte,MG,delivered,2017-11-18 19:28:06,5.0,27.20,2017-11,17,3,9,12,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,SP,2c9e548be18521d1c43cde1c582c6de8,mogi das cruzes,SP,delivered,2018-02-13 21:18:39,5.0,8.72,2018-02,61,0,1,1,False


In [253]:
fig = go.Figure(
    go.Histogram(
        x = data_orders_["review_score"],
        histfunc = "count",
        texttemplate = "%{y}",
        name = "Review score",
        textfont_size = 12,
        textposition = "outside",
        hovertemplate = "Review score: %{x}<br>No. of orders: %{y}",
        xbins = dict (
            start = -1.0,
            end = 5.0,
            size = 1.0
        )
    )
)

fig.update_layout(
    title=dict(
        text="<b>Customer review<b>",
        font=dict(size=12, family="Arial", color="black")
    ),
    plot_bgcolor = "white",
    xaxis = dict(title = "Review score"),
    yaxis = dict(showticklabels = False)
)

fig.add_annotation(
    text="Created By Thuan Dao.",
    xref="paper", yref="paper",
    x=1, y=-0.2,
    showarrow=False,
    font=dict(size=10, color="gray", family="Arial")
)
fig.show()

<font color = '#e3f56e'>
Insight

* Ratings 5 & 4 dominate (~19.3K each) → majority of customers are satisfied.
* Rating 1 is also high (12.05K) → clear segment of dissatisfied customers.
* Middle ratings (2 & 3) are lower → feedback is polarized.
* Rating 0 is rare (799) but worth investigating → may indicate system issues or very poor experience.
</font>

### Shipment Complettion Ratio

In [254]:
# Count the number of orders for each order status
data_orders_status = (
    data_orders_.groupby("order_status")["order_id"]
    .count()  # count orders per status
    .reset_index()
)

# Calculate the percentage of each order status relative to total orders
data_orders_status["%ratio"] = data_orders_status["order_id"] / data_orders_status["order_id"].sum() * 100

display(data_orders_status.head())

,order_status,order_id,%ratio
0,approved,2,0.001962
1,canceled,628,0.616170
2,created,5,0.004906
3,delivered,98937,97.073195
4,invoiced,318,0.312009


In [255]:
# Count total shipments (all orders)
total_shipments = data_orders_["order_id"].nunique()

# Count total cancelled shipments
total_cancelled_shipments = data_orders_[data_orders_["order_status"] == "canceled"]["order_id"].nunique()

# Calculate and print the cancelled shipment ratio
print(f"Cancelled shipment ratio is {(total_cancelled_shipments / total_shipments) * 100:.2f}%")

Cancelled shipment ratio is 0.63%


In [256]:
# Count total delivered shipments
total_delivered_shipments = data_orders_[data_orders_["order_status"] == "delivered"]["order_id"].nunique()

# Count total on-time delivered shipments (is_late == False)
total_on_time_delivered_shipments = data_orders_[
    (data_orders_["order_status"] == "delivered") & (data_orders_["is_late"] == False)
]["order_id"].nunique()

# Calculate and print the on-time delivery ratio
print(f"On-time delivered shipment ratio is {(total_on_time_delivered_shipments / total_delivered_shipments) * 100:.2f}%")

On-time delivered shipment ratio is 91.89%


In [257]:
# Calculate the average approval time (in minutes) per order, excluding orders with 0 approval time
avg_confirmed_mins = (
    data_orders_.query("approved_mins != 0")                    # filter out orders with 0 approval time
    .groupby("order_id")                                       # group by each order
    .agg({"approved_mins": "mean"})["approved_mins"]           # compute mean approval time per order
    .mean()                                                    # average across all orders
)

# Calculate the average delivery time (in days) per order, excluding orders with 0 delivery days
avg_delivery_days = (
    data_orders_.query("delivery_days != 0")                   # filter out orders with 0 delivery days
    .groupby("order_id")                                       # group by each order
    .agg({"delivery_days": "mean"})["delivery_days"]          # compute mean delivery time per order
    .mean()                                                    # average across all orders
)

# Print the results
print(f"For each order, it takes approximately {avg_confirmed_mins:.0f} minute(s) on average for an order to be approved.")
print(f"For each order, it takes approximately {avg_delivery_days:.0f} day(s) on average to deliver to customer after approved.")

For each order, it takes approximately 633 minute(s) on average for an order to be approved.
For each order, it takes approximately 11 day(s) on average to deliver to customer after approved.


In [258]:
# Aggregate monthly order counts by order status
data_orders_year_month = (
    data_orders_[["order_id", "year_month", "order_status"]]  # select relevant columns
    .drop_duplicates()                                         # remove duplicate orders
    .groupby(["year_month", "order_status"])                  # group by month and status
    .agg(orders_count=("order_status", "count"))             # count orders per status
    .reset_index()
    # Pivot to have one row per month and one column per order_status
    .pivot(index="year_month", columns="order_status", values="orders_count")
    .reset_index()                                           # reset index for clean DataFrame
    .fillna(0)                                               # fill missing counts with 0
)

display(data_orders_year_month.head())

order_status,year_month,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
0,2016-09,0.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0
1,2016-10,0.0,24.0,0.0,265.0,18.0,2.0,8.0,7.0
2,2016-12,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,2017-01,0.0,3.0,0.0,750.0,12.0,9.0,16.0,10.0
4,2017-02,1.0,17.0,0.0,1653.0,11.0,32.0,21.0,45.0


In [259]:
year_month_list = data_orders_year_month["year_month"].astype(str).to_list()
order_status_list = list(set(data_orders_["order_status"]))
fig = go.Figure()

for order_status in order_status_list:
    fig.add_trace(go.Scatter(
        x = year_month_list,
        y = data_orders_year_month[order_status].to_list(),
        name = order_status,
        mode = "lines",
        line = dict (width = 0.5),
        stackgroup = "one" # define stack group
    ))

fig.update_layout(
    title=dict(
        text="<b>Orders by Status<b>",
        font=dict(size=12, family="Arial", color="black")
    ),
    plot_bgcolor = "white",
    hovermode = "x unified"
)

fig.add_annotation(
    text="Created By Thuan Dao.",
    xref="paper", yref="paper",
    x=1.1, y=-0.2,
    showarrow=False,
    font=dict(size=10, color="gray", family="Arial")
)
fig.show()

<font color = '#e3f56e'>
Insight

* **Delivered orders dominate** → most orders are successfully completed.
* **Created, processing, approved, shipped, invoiced, canceled, unavailable** → very few orders in these statuses, minimal impact.
* **Trend over time**: Delivered orders steadily increased from early 2017, peaking in early 2018, then slightly declining by mid-2018.
* Focus on maintaining high delivery rate and investigate any causes of canceled/unavailable orders to improve fulfillment efficiency.

</font>

In [260]:
# Initialize a new column to store total orders per month
data_orders_year_month["total_orders"] = 0

# Sum the counts of all order statuses to get total orders per month
for status in set(data_orders_["order_status"]):
    data_orders_year_month["total_orders"] += data_orders_year_month[status]

# Calculate average monthly orders
avg_orders_by_month = data_orders_year_month["total_orders"].mean()
print(f"On average, {avg_orders_by_month:.0f} orders created are per month.")

# Calculate total orders in the dataset
total_orders = data_orders_year_month["total_orders"].sum()

# Calculate total days covered by the dataset
total_days = (data_orders_["order_purchase_timestamp"].max() - data_orders_["order_purchase_timestamp"].min()).days
print(f"On average, {total_orders/total_days:.0f} orders are created per day.")

# Calculate total hours covered and average orders per hour
total_hours = total_days * 24
print(f"On average, {total_orders/total_hours:.0f} orders are created per hour.")

On average, 3978 orders created are per month.
On average, 129 orders are created per day.
On average, 5 orders are created per hour.


In [261]:
# Create a temporary DataFrame with order IDs and purchase timestamps
temp = (
    data_orders_[["order_id", "order_purchase_timestamp"]]
    .copy()
    .sort_values(by="order_purchase_timestamp")  # sort orders chronologically
    .drop_duplicates()                            # ensure each order is counted once
)

# Calculate the time difference (in minutes) between consecutive orders
temp["order_purchase_diff"] = (
    temp["order_purchase_timestamp"].diff()       # time difference between consecutive orders
    .dt.total_seconds() / 60                       # convert seconds to minutes
).fillna(0).astype(int)                           # fill first NaN with 0 and convert to int

# Compute the average time difference between orders
avg_diff = temp["order_purchase_diff"].mean()

# Print the result
print(f"On average, after {avg_diff:.0f} minutes an order would be created.")

On average, after 11 minutes an order would be created.


### Evaluation by sellers

In [262]:
# Aggregate shipping performance metrics by route (seller city/state -> customer city/state)
data_shipping = (
    data_orders_[[
        "order_id", "seller_city", "order_status", "seller_state",
        "customer_city", "customer_state", "freight_value",
        "to_carrier_days", "to_customer_days", "delivery_days"
    ]]
    .query('order_status == "delivered"')  # consider only delivered orders
    .groupby(["seller_city", "seller_state", "customer_city", "customer_state"])
    .agg(
        orders_num_same_route=("order_id", "nunique"),  # number of unique orders on the same route
        avg_freight_value=("freight_value", "mean"),    # average freight cost
        avg_delivery_days=("delivery_days", "mean"),    # average delivery days
    )
    .reset_index()
)

# Define desired data types for the aggregated metrics
obj_type = {
    "orders_num_same_route": "int",
    "avg_freight_value": "float",
    "avg_delivery_days": "int",
}

# Apply data types
data_shipping = data_shipping.astype(obj_type)

display(data_shipping.head())

,seller_city,seller_state,customer_city,customer_state,orders_num_same_route,avg_freight_value,avg_delivery_days
0,abadia de goias,GO,sobral,CE,1,43.41,24
1,afonso claudio,ES,belem,PA,1,29.62,20
2,afonso claudio,ES,franca,SP,1,17.19,11
3,afonso claudio,ES,macae,RJ,1,15.56,6
4,afonso claudio,ES,niteroi,RJ,1,17.43,7


In [263]:
# Aggregate seller performance metrics
data_sellers = (
    data_orders_[[
        "order_id", "seller_id", "seller_city", "seller_state",
        "order_status", "is_late", "review_score", "approved_mins"
    ]]
    .drop_duplicates()  # remove duplicate rows for clean aggregation
    .groupby(["seller_id", "seller_city", "seller_state"])
    .apply(lambda x: pd.Series({
        'total_orders': x['order_id'].nunique(),  # total orders handled by the seller
        'delivered_orders': x.query("order_status == 'delivered'")['order_id'].nunique(),  # delivered orders
        'cancelled_orders': x.query("order_status == 'canceled'")['order_id'].nunique(),   # canceled orders
        'late_delivery': x.query("is_late == True")['order_id'].nunique(),  # late deliveries
        'avg_approved_mins': x['approved_mins'].mean(),  # average approval time in minutes
        'review_score': x.query("review_score != 0")['review_score'].mean(),  # average non-zero review score
    }))
    # Compute delivered and cancelled ratios as percentages
    .assign(
        delivered_ratio=lambda x: round(x["delivered_orders"] / x["total_orders"] * 100, 2),
        cancelled_ratio=lambda x: round(x["cancelled_orders"] / x["total_orders"] * 100, 2),
    )
    .reset_index()  # reset index for a clean DataFrame
)

display(data_sellers.head())

,seller_id,seller_city,seller_state,total_orders,delivered_orders,cancelled_orders,late_delivery,avg_approved_mins,review_score,delivered_ratio,cancelled_ratio
0,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP,3.0,3.0,0.0,0.0,800.666667,3.666667,100.00,0.0
1,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES,200.0,195.0,0.0,13.0,591.290000,3.984772,97.50,0.0
2,001e6ad469a905060d959994f1b41e4f,sao goncalo,RJ,1.0,0.0,1.0,0.0,14.000000,1.000000,0.00,100.0
3,002100f778ceb8431b7a1020ff7ab48f,franca,SP,51.0,50.0,0.0,9.0,1376.313725,3.901961,98.04,0.0
4,003554e2dce176b5555353e4f3555ac8,goiania,GO,1.0,1.0,0.0,0.0,18.000000,5.000000,100.00,0.0


In [264]:
# Aggregate product-level sales per seller with category translation
data_items = (
    df_order_items
    # Merge product category names
    .merge(df_products[["product_id", "product_category_name"]], how="left", on="product_id")
    # Merge translated category names
    .merge(df_product_category_name_translation, how="left", on="product_category_name")
    # Group by seller, product, and product category (English)
    .groupby(["seller_id", "product_id", "product_category_name_english"])
    .agg(
        product_count=("product_id", "count")  # count number of times each product was sold by the seller
    )
    # Sort by number of products sold descending
    .sort_values(by="product_count", ascending=False)
    .reset_index()
)

display(data_items.head())

,seller_id,product_id,product_category_name_english,product_count
0,955fee9216a65b617aa5c0531780ce60,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527
1,1f50f920176fa81dab994f9023523100,422879e10f46682990de24d770e7f83d,garden_tools,484
2,4a3ca9315b744ce9f8e9374361493884,99a4788cb24856965c36a24e339b6058,bed_bath_table,482
3,1f50f920176fa81dab994f9023523100,389d119b48cf3043d311335e499d9c6b,garden_tools,392
4,1f50f920176fa81dab994f9023523100,368c6c730842d78016ad823897a372db,garden_tools,388


In [265]:
def suggest(city: str, state: str, product_category, limit=10, product_id=None):
    """
    Suggest sellers based on customer's city and state and search on a product category (or specific product).
    
    Parameters:
    - city: customer city
    - state: customer state
    - product_category: product category (English)
    - limit: number of suggested sellers to return (default=10)
    - product_id: specific product to filter (optional)
    
    Returns:
    - DataFrame of suggested sellers with detailed metrics
    """

    # Filter sellers who sell the specific product or product category
    if product_id is not None:
        df_product = data_items[
            (data_items["product_id"] == product_id) &
            (data_items["product_category_name_english"] == product_category)
        ]
    else:
        df_product = data_items[
            data_items["product_category_name_english"] == product_category
        ]

    # Aggregate products sold per seller
    df_product = df_product.groupby("seller_id").apply(
        lambda x: pd.Series({
            "products_sold": x[["product_id", "product_count"]].to_dict("records")
        })
    )

    # Merge with seller performance metrics
    df = df_product.merge(data_sellers, how="left", on="seller_id")

    # Filter sellers who ship to the customer's city/state
    df_location = data_shipping[
        (data_shipping["customer_city"] == city) &
        (data_shipping["customer_state"] == state)
    ]
    df = df.merge(df_location, how="inner", on=["seller_city", "seller_state"])

    # Sort sellers by:
    # 1) highest delivered ratio
    # 2) lowest average delivery days
    # 3) lowest average approval time
    df = df.sort_values(
        by=["delivered_ratio", "avg_delivery_days", "avg_approved_mins"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    # Select relevant columns to return
    df = df[[
        "seller_id", "seller_city", "seller_state", "customer_city", "customer_state",
        "products_sold", "total_orders", "delivered_orders", "cancelled_orders", "late_delivery",
        "avg_approved_mins", "review_score", "delivered_ratio", "cancelled_ratio",
        "orders_num_same_route", "avg_freight_value", "avg_delivery_days"
    ]]

    return df

suggest(product_category="garden_tools", city="sao goncalo", state="RJ").head()

,seller_id,seller_city,seller_state,customer_city,customer_state,products_sold,total_orders,delivered_orders,cancelled_orders,late_delivery,avg_approved_mins,review_score,delivered_ratio,cancelled_ratio,orders_num_same_route,avg_freight_value,avg_delivery_days
0,0a198e95d32b1be2da9424c962a6ebfa,contagem,MG,sao goncalo,RJ,"[{'product_id': 'f0788219c3d63c6183bbca7699ca9608', 'product_count': 1}]",1.0,1.0,0.0,0.0,27.000000,5.00,100.0,0.0,3,28.963333,4
1,2aa3443d7bf9d9bb11133f420d75e083,rio de janeiro,RJ,sao goncalo,RJ,"[{'product_id': '3630657a252ca88c694edfa53f2ad9f2', 'product_count': 1}]",10.0,10.0,0.0,2.0,243.500000,3.00,100.0,0.0,12,9.973333,4
2,a4bd6e9adf39b63f43dc545d3ca1f53d,rio de janeiro,RJ,sao goncalo,RJ,"[{'product_id': 'e6baba6c7819d44817a76305e082d682', 'product_count': 1}, {'product_id': 'c511bbd742df73d5b2ba4594d4901f2b', 'product_count': 1}, {'product_id': '902ef94ec6b84c1abdc30ac2877a0e95', 'product_count': 1}]",4.0,4.0,0.0,0.0,384.250000,4.50,100.0,0.0,12,9.973333,4
3,7901646fdd36a55f564ffaf2dbccaaf7,rio de janeiro,RJ,sao goncalo,RJ,"[{'product_id': '636598095d69a5718e67d2c9a3c7dde6', 'product_count': 4}, {'product_id': '6e7df7f4622d4360261995dbd5e787d0', 'product_count': 3}, {'product_id': '8374b39a15882a19ce4558f13064b55c', 'product_count': 2}, {'product_id': 'b6397895a17ce86decd60b898b459796', 'product_count': 1}, {'product_id': '5a968ab149dac747332323c42b49e30a', 'product_count': 1}, {'product_id': '57e7f9befd08f9e14bc4545abf760d0e', 'product_count': 1}, {'product_id': '03ac940e93916395ea0483161cc84d5c', 'product_count': 1}, {'product_id': 'ff1f1de0f05e0ec5b2d721e2fb425b0c', 'product_count': 1}]",22.0,22.0,0.0,3.0,607.045455,4.50,100.0,0.0,12,9.973333,4
4,8a9260f2b0340411d6d2a56bcf4f7378,contagem,MG,sao goncalo,RJ,"[{'product_id': 'b800d7bb8cd5a7093dd099a367d1dde2', 'product_count': 2}]",8.0,8.0,0.0,1.0,630.875000,4.75,100.0,0.0,3,28.963333,4


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #FFFFFF; 
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7); 
        font-weight: bold; 
        margin-bottom: 5px; 
        font-size: 28px; 
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        🙏 Thanks for Reading! 🚀
    </h1>
    <p style="color: #ffffff; font-size: 18px; text-align: center;">
        Happy Coding! 🙌😊
    </p>
</div>
